In [28]:
from qiskit_ibm_runtime.fake_provider import FakeAlgiers
import pandas as pd
import numpy as np

In [30]:
backend = FakeAlgiers()

print(f"Backend: {backend.name}")
print(f"Number of qubits: {backend.num_qubits}")
print(f"Portas Nativas: {backend.operation_names}")

n_qubits = 5

# 1. T1 e T2 (Médias dos primeiros 5 qubits em nanossegundos)
t1_values = [backend.target.qubit_properties[i].t1 for i in range(n_qubits)]
t2_values = [backend.target.qubit_properties[i].t2 for i in range(n_qubits)]

# 2. Durações das Portas (ns)
# Pegamos a duração da porta 'sx' (1q) e 'cx' (2q)
gate_1q_ns = backend.target['sx'][(0,)].duration * 1e9
gate_2q_ns = backend.target['cx'][(0, 1)].duration * 1e9

# 3. Erro de Despolarização (p) - Média dos pares conectados entre os 5 qubits
# O FakeTorino é um Heavy-Hex, precisamos ver quais pares (0-4) existem
connected_pairs = []
for pair in backend.target['cx']:
    if all(q < n_qubits for q in pair):
        connected_pairs.append(pair)

p_values = [backend.target['cx'][pair].error for pair in connected_pairs]

# 4. Erro de Leitura (Readout) - Média dos 5 qubits
readout_errors = [backend.target['measure'][(i,)].error for i in range(n_qubits)]

# --- Montando o dicionário final ---
noise_params = {
    "p": np.mean(p_values),
    "t1": np.mean(t1_values) * 1e9, # Convertendo s para ns
    "t2": np.mean(t2_values) * 1e9, # Convertendo s para ns
    "gate_1q": gate_1q_ns,
    "gate_2q": gate_2q_ns,
    "readout_error": np.mean(readout_errors) # Informação extra para sua validação
}

print("Parâmetros de Ruído Extraídos (FakeTorino):")
for k, v in noise_params.items():
    print(f"{k}: {v:.6f}")

Backend: fake_algiers
Number of qubits: 27
Portas Nativas: ['sx', 'rz', 'x', 'if_else', 'delay', 'for_loop', 'id', 'switch_case', 'cx', 'measure', 'reset']
Parâmetros de Ruído Extraídos (FakeTorino):
p: 0.006471
t1: 156847.124085
t2: 188818.139488
gate_1q: 35.555556
gate_2q: 259.555556
readout_error: 0.007440


In [4]:
# Extraindo dados de todos os qubits para uma tabela
data = []
for i in range(backend.num_qubits):
    q_props = backend.qubit_properties(i)
    # Nota: Em backends V2, alguns dados detalhados vêm do target
    data.append({
        "Qubit": i,
        "T1": backend.target.qubit_properties[i].t1,
        "T2": backend.target.qubit_properties[i].t2,
    })

df = pd.DataFrame(data)
print(df.head())plt.style.use('default')
plt.plot(loss_history_noise)
plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.title("noisy Loss convergence")
plt.show()

plt.plot(param_history_noise)
plt.xlabel("Iterations")
plt.ylabel("Parameters")
plt.title("Parameters convergence")
plt.show()


   Qubit        T1        T2
0      0  0.000224  0.000315
1      1  0.000266  0.000204
2      2  0.000170  0.000152
3      3  0.000254  0.000210
4      4  0.000156  0.000190
